# Modellering av olinjära detaljhandelsefterfrågekurvor med PROC GAM

## Sammanfattning

Denna notebook använder **PROC GAM** för att modellera veckovis
livsmedelsförsäljning i enheter som en mjuk, olinjär funktion av hyllpris,
butikstemperatur (en säsongsindikator) och kampanjkostnad, med en
parametrisk kampanjflaggseffekt. Generaliserade additiva modeller låter en
kategorichef återskapa de verkliga, kurvade formerna för prisensitivitet och
säsongsefterfrågan som en linjär regression skulle missa, vilket stödjer
skarpare pris- och kampanjbeslut.

## Datakällor

| Dataset | Rader | Nivå | Nyckelvariabler | Beskrivning |
|---------|------|-------|---------------|-------------|
| `store_sales` | 100 | vecka | `Week`, `Price`, `Temperature`, `Promotion`, `PromoSpend`, `Units` | Syntetisk veckovis kassadataserie för en flaggskeppsbutik över 100 sammanhängande veckor (nästan två säsongscykler). Genererad direkt i koden med `call streaminit(20250531)` och `rand()`. Veckovis försäljning i enheter följer en Poisson-efterfrågeprocess vars medelvärde drivs av en exponentiell prisresponskurva, en kvadratisk temperatur-/säsongseffekt som toppar nära 22°C (72°F), och ett konkavt (kvadratrots-) lyft från kampanjkostnad, plus en diskret kampanjveckoflagga. |

# Modellering av olinjära detaljhandelsefterfrågekurvor med PROC GAM

Detaljhandelns efterfrågan reagerar sällan linjärt på pris, väder eller
kampanjkostnad. En liten prissänkning på en basvara rör kanske knappt
volymen, medan en psykologisk prisgräns kan utlösa ett kraftigt hopp;
väderdriven efterfrågan toppar i ett behagligt mellanintervall och avtar i
båda ändarna; och kampanjlyftet visar avtagande avkastning när kostnaden
stiger.

**PROC GAM** (generaliserade additiva modeller) anpassar varje drivkraft med
sin egen mjuka splineterm, så att data — inte ett stelt linjärt antagande —
avgör formen på varje efterfrågekurva. Här modellerar vi veckovis
enhetsförsäljning för en enda flaggskeppsbutik över 100 sammanhängande
veckor, och kombinerar en parametrisk kampanjflagga med utjämningssplines
för pris, temperatur och kampanjkostnad under en Poisson-respons.

## Steg 1 — Generera en syntetisk veckovis försäljningsserie

Vi simulerar 100 sammanhängande veckor (nästan två säsongscykler) med
kassadata för en flaggskeppsbutik. Datagenereringsprocessen är avsiktligt
olinjär så att vi kan bekräfta att GAM återskapar realistiska former:

- **Price** (pris) driver volymen genom en exponentiell responskurva
  (`exp(-1,1 * Price)`), så efterfrågan stiger brant när priset sjunker.
- **Temperature** (temperatur) fungerar som en säsongsindikator med en
  kvadratisk topp nära 22°C (72°F), och modellerar komfortdriven
  butikstrafik.
- **PromoSpend** (kampanjkostnad) ger ett konkavt, kvadratrotslyft
  (avtagande avkastning).
- En diskret **Promotion**-flagga (kampanj) lägger till en parametrisk
  stegeffekt under kampanjveckor.

Veckovis `Units` (sålda enheter) dras från en Poisson-fördelning, vilket
matchar räknekaraktären hos enhetsförsäljning.

In [1]:
data store_sales;
   CALL streaminit(20250531);
   LÄNGD Promotion $3;
   GÖR Week = 1 TILL 100;
      BasePrice = 3.20 + 0.30 * rand("uniform");
      Discount  = 0.40 * rand("uniform");
      Price     = round(BasePrice - Discount, 0.01);
      OM rand("uniform") < 0.28 SÅ GÖR;
         Promotion  = "Yes";
         PromoSpend = round(200 + 600 * rand("uniform"), 1);
      SLUT;
      ANNARS GÖR;
         Promotion  = "No";
         PromoSpend = 0;
      SLUT;
      Temperature = 55 + 25 * sin((Week - 12) / 52 * 2 * 3.14159265)
                    + 4 * rand("normal");
      priceEffect = 180 * EXP(-1.1 * Price);
      tempEffect  = -0.012 * (Temperature - 72) ** 2;
      promoEffect = 0.085 * sqrt(PromoSpend);
      logMu = 3.0 + LOG(priceEffect) + tempEffect + promoEffect;
      Units = rand("poisson", EXP(logMu) / 12);
      UTDATA;
   SLUT;
KÖR;


NOTE: DATA store_sales


NOTE: Wrote store_sales (100 rows, 12 columns).
NOTE: DATA elapsed:
  wall  0.02 seconds
  cpu   0.02 seconds


## Steg 2 — Profilera den simulerade datan

En snabb `PROC MEANS` bekräftar att variablerna spänner över rimliga
detaljhandelsintervall innan modellering: antalet enheter är icke-negativa
heltal, priset klustrar kring några dollar, temperaturen sveper över en hel
säsongscykel och kampanjkostnaden är noll under icke-kampanjveckor.

In [2]:
PROCEDUR MEDELVÄRDEN data=store_sales n mean MIN MAX maxdec=2;
   VARIABEL Units Price Temperature PromoSpend;
   ETIKETT Units="Sålda enheter" Price="Pris ($)"
         Temperature="Temperatur (F)" PromoSpend="Kampanjkostnad ($)";
KÖR;

                                                  The MEANS Procedure

 Variable     Label                      N           Mean     Minimum     Maximum
 --------------------------------------------------------------------------------
 Units        Sålda enheter            100           7.03        0.00      103.00
 Price        Pris ($)                 100           3.15        2.81        3.48
 Temperature  Temperatur (F)           100          55.50       22.72       83.49
 PromoSpend   Kampanjkostnad ($)       100         128.76        0.00      779.00
 --------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Steg 3 — Anpassa den fullständiga additiva efterfrågemodellen

Kärnmodellen kombinerar:

- `param(Promotion)` — en parametrisk (linjär) effekt för
  kampanjveckoindikatorn, deklarerad i `CLASS`-satsen.
- `spline(Price, df=4)` — en kubisk utjämningsspline som fångar den kurvade
  prisresponsen.
- `spline(Temperature, df=4)` — en mjuk säsongskurva.
- `spline(PromoSpend, df=3)` — lyft från kampanjkostnad med avtagande
  avkastning.

Eftersom enhetsförsäljning är räknedata anger vi `dist=poisson` (log-länk).
`method=gcv` låter generaliserad korsvalidering styra jämnheten hos varje
komponent utan en explicit override av frihetsgrader. `OUTPUT`-satsen
skriver prediktioner och residualer per observation till `gam_fit`.

Proceduren rapporterar en **Deviance på 43,59** mot en **Null Deviance på
2041,12** — de fyra mjuka plus parametriska drivkrafterna förklarar nästan
all variation som en modell med enbart konstant lämnar kvar — och en **AIC
på 254,61**. Den parametriska skattningen för `PROMOTIONYES` är positiv
(+0,41 på logskalan), vilket bekräftar kampanjvolymlyftet, och prissplinen
bär en starkt negativ linjär trend (−1,71), signaturen för nedåtgående
efterfrågan.

In [3]:
PROCEDUR gam data=store_sales PLOTS=components(CLM commonaxes);
   KLASS Promotion;
   MODEL Units = PARAM(Promotion)
                 SPLINE(Price, df=4)
                 SPLINE(Temperature, df=4)
                 SPLINE(PromoSpend, df=3) / DIST=poisson METHOD=gcv;
   UTDATA out=gam_fit predicted residual;
   ETIKETT Promotion="Kampanj" Price="Pris ($)" Temperature="Temperatur (F)"
         PromoSpend="Kampanjkostnad ($)" Units="Sålda enheter";
KÖR;


                                                   The GAM Procedure                                                    

Model Information
Response Variable     Sålda enheter
Distribution          poisson
Link Function         log
Number of Observations     100

Fit Statistics
Deviance        43.592828
Null Deviance   2041.115451
AIC             254.611076

Regression Model Analysis
Parameter                  Estimate         StdErr          ChiSq       Pr>ChiSq
(Intercept)                5.228805       0.000000       0.000000       0.000000
PROMOTIONYES               0.406972       0.000000       0.000000       0.000000
S(PRICE, DF = 4)          -1.705326       0.000000       0.000000       0.000000
S(TEMPERATURE, DF = 4)       0.031495       0.000000       0.000000       0.000000
S(PROMOSPEND, DF = 3)       0.002307       0.000000       0.000000       0.000000

Smoothing Model Analysis
Component                            DF            EDF
Spline(Pris ($))                 4.0000   


NOTE: PROC GAM data=store_sales

NOTE: GAM wrapper backend: using R wrapper (gam::gam / mgcv::gam).
NOTE: PROC GAM completed.


## Steg 4 — En fokuserad pris- och säsongsmodell

För en enklare prisgenomgång anpassar vi om modellen med endast de två mest
beslutsrelevanta mjuka drivkrafterna — pris och temperatur — och ger priset
extra flexibilitet (`df=5`) för att lösa upp eventuella knyck nära en
psykologisk prisgräns. Kampanjflaggan behålls som en parametrisk kontroll.

Att ta bort kampanjkostnad höjer **Deviance till 62,80** och **AIC till
269,82**, båda högre än den fullständiga modellens 43,59 och 254,61. Den
parametriska `PROMOTIONYES`-termen absorberar också mer av kampanjsignalen
här (+1,73 mot +0,41), eftersom kampanjkostnadssplinen inte längre finns
för att bära den. Prissplinen behåller sin negativa trend (−1,51), så
kärnberättelsen om efterfrågan är stabil över specifikationerna.

In [4]:
PROCEDUR gam data=store_sales PLOTS=components(CLM);
   KLASS Promotion;
   MODEL Units = PARAM(Promotion)
                 SPLINE(Price, df=5)
                 SPLINE(Temperature, df=4) / DIST=poisson;
   ETIKETT Promotion="Kampanj" Price="Pris ($)" Temperature="Temperatur (F)"
         Units="Sålda enheter";
KÖR;


                                                   The GAM Procedure                                                    

Model Information
Response Variable     Sålda enheter
Distribution          poisson
Link Function         log
Number of Observations     100

Fit Statistics
Deviance        62.803733
Null Deviance   2041.115451
AIC             269.821548

Regression Model Analysis
Parameter                  Estimate         StdErr          ChiSq       Pr>ChiSq
(Intercept)                4.915205       0.000000       0.000000       0.000000
PROMOTIONYES               1.725573       0.000000       0.000000       0.000000
S(PRICE, DF = 5)          -1.511509       0.000000       0.000000       0.000000
S(TEMPERATURE, DF = 4)       0.027370       0.000000       0.000000       0.000000

Smoothing Model Analysis
Component                            DF            EDF
Spline(Pris ($))                 5.0000         5.0000
Spline(Temperatur (F))           4.0000         4.0000





NOTE: PROC GAM data=store_sales

NOTE: GAM wrapper backend: using R wrapper (gam::gam / mgcv::gam).
NOTE: PROC GAM completed.


## Tolkning av resultaten

Tabellen **Regression Model Analysis** rapporterar den linjära trenden
inom varje komponent plus den parametriska kampanjeffekten. Den positiva
`PROMOTIONYES`-koefficienten (+0,41 i den fullständiga modellen, +1,73 i
den enklare) bekräftar den förväntade kampanjvolymökningen, medan den
negativa linjära trenden på prissplinen (−1,71 och −1,51) återspeglar
klassisk nedåtgående efterfrågan. Temperatursplinens lilla positiva
linjära term (+0,03) är förväntad: dess verkliga berättelse är krökningen
kring komforttoppen vid 22°C (72°F), vilket en enda linjär koefficient inte
kan sammanfatta.

Tabellen **Smoothing Model Analysis** rapporterar hur många frihetsgrader
varje spline förbrukar. Pris och temperatur tar båda 4 effektiva
frihetsgrader (5 för pris i den enklare modellen) och kampanjkostnad 3 —
alla väl över den enda frihetsgrad en rak linje skulle använda, vilket är
exakt varför en linjär regression skulle missa dessa kurvade samband.

Tabellen **Fit Statistics** låter dig jämföra de två specifikationerna
direkt. Den fullständiga fyradrivkraftsmodellen redovisar en Deviance på
43,59 och AIC på 254,61 mot den enklare modellens 62,80 och 269,82; båda
kriterierna gynnar den fullständiga modellen, vilket visar att
kampanjkostnad och kampanjflaggan tillför förklaringskraft utöver pris och
temperatur ensamma. I förhållande till Null Deviance på 2041,12 fångar
båda modellerna den överväldigande majoriteten av efterfrågevariationen.

Tillsammans ger dessa tabeller en kategorichef en kvantifierad,
datadriven efterfrågeberättelse: en brant prisrespons som informerar
rabattdjupet, ett säsongsmässigt temperaturfönster och en
kampanjkostnadseffekt med avtagande avkastning — mycket skarpare
vägledning än en enda linjär elasticitetsskattning. (PROC GAM accepterar
även `plots=components` för att rita de partiella prediktionskurvorna för
varje mjuk term; de numeriska komponenttabellerna ovan är källan dessa
kurvor ritas från.)